[`runner.py`](https://github.com/open-mmlab/mmengine/blob/main/mmengine/runner/runner.py#L451)

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
from torchvision.utils import make_grid

In [2]:
from __future__ import annotations

import os
import time
import copy
from functools import partial
from typing import Optional, Union

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch import distributed as torch_dist

import numpy as np

from mmengine import Config, DictAction

import sys

sys.path.append('../../../../')

%load_ext autoreload
%autoreload 2
    
from computer_vision.slowfast.mmaction.utils import SampleList
from computer_vision.slowfast.mmaction.models.recognizers.recognizer3d import Recognizer3D

from computer_vision.slowfast.mmaction.datasets.transforms.loading import DecordInit, SampleFrames, DecordDecode
from computer_vision.slowfast.mmaction.datasets.transforms.processing import Resize, RandomCrop, CenterCrop, ThreeCrop, RandomResizedCrop, Flip
from computer_vision.slowfast.mmaction.datasets.transforms.formatting import FormatShape, PackActionInputs
from computer_vision.slowfast.mmengine.dataset.base_dataset import Compose
from computer_vision.slowfast.mmengine.dataset.utils import pseudo_collate, worker_init_fn
from computer_vision.slowfast.parameter_parser import parser, merge_args


from computer_vision.slowfast.mmengine.runner.utils import set_random_seed
from computer_vision.slowfast.mmengine.runner.loops import EpochBasedTrainingLoop, ValLoop
from computer_vision.slowfast.mmengine.runner.runner import Runner
from computer_vision.slowfast.mmaction.datasets.video_dataset import VideoDataset
from computer_vision.slowfast.mmengine.dataset.sampler import DefaultSampler
from computer_vision.slowfast.mmengine.optim.optimizer.optimizer_wrapper import OptimWrapper
from computer_vision.slowfast.mmengine.optim.scheduler.lr_scheduler import LinearLR, CosineAnnealingLR
from computer_vision.slowfast.mmengine.evaluator.evaluator import Evaluator

In [3]:
config="../../config/slowfast_r50_8xb8-4x16x1-256e_kinetics400-rgb.py"

data_dirpath='D:/data/UCF101'
# root=f'{data_dirpath}/UCF-101'
# annotation_path=f'{data_dirpath}/UCF101TrainTestSplits-RecognitionTask/trainlist01.txt'
annotation_path='UCF101TrainTestSplits-RecognitionTask/trainlist01.txt'
metadata_path=f'{data_dirpath}/metadata.pt'

mini_train=False
if not mini_train:
    output_dirpath="D:/results/ucf101/mmaction2-slowfast/train"
    arguments= f"""-d {data_dirpath} -a {annotation_path} 
    """ # --use-cutmix-mixup
else:
    output_dirpath="D:/results/ucf101/mmaction2-slowfast/mini_train"
    arguments= f"""-d {data_dirpath} -a {annotation_path}
    """ # --use-cutmix-mixup --time 18
    
arguments+=f"""--data-prefix UCF-101 {config} --work-dir {output_dirpath} --auto-scale-lr --seed 1"""

args=parser.parse_args(arguments.split())
cfg=Config.fromfile(args.config)

cfg=merge_args(cfg, args)

# create default Runner, `runner = Runner.from_cfg(cfg)`  and call `runnner.train()`

recognizer=Recognizer3D(backbone=cfg.model.backbone, cls_head=cfg.model.cls_head, train_cfg=None, test_cfg=None,
                data_preprocessor=cfg.model.data_preprocessor)
runner=Runner(model=recognizer, work_dir=cfg.work_dir, cfg=cfg)

cfg.filename
In resnet3d_slowfast.ResNet3dSlowFast.__init__ slow_pathway={'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': True, 'conv1_kernel': (1, 7, 7), 'dilations': (1, 1, 1, 1), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'inflate': (0, 0, 1, 1), 'norm_eval': False, 'speed_ratio': 8, 'channel_ratio': 8} 
In resnet3d_slowfast.ResNet3dSlowFast.__init__ fast_pathway={'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': False, 'base_channels': 8, 'conv1_kernel': (5, 7, 7), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'norm_eval': False} 
In ResNet3dPathway._calculate_lateral_inplanes: depth=50, expansion=4, base_channels=64
stage 0 ----------
	planes=64, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 1 ----------
	planes=256, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 2 ----------
	planes=512, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 3 ----------
	planes=1024, self.lat

[`build_dataloader`](https://github.com/open-mmlab/mmengine/blob/main/mmengine/runner/runner.py#L1328)

In [4]:
# def train(self)->nn.Module:
"""Launch training
Returns:
    (nn.Module): The model after training
Reference: https://github.com/open-mmlab/mmengine/blob/main/mmengine/runner/runner.py#L1702
"""
# build train loop

pipeline=[DecordInit(io_backend='disk'),
            SampleFrames(clip_len=32, frame_interval=2, num_clips=1),
            DecordDecode(),
            Resize(scale=(-1, 256), keep_ratio=True, interpolation='bilinear', lazy=False),
            RandomResizedCrop(),
            Resize(scale=(224, 224), keep_ratio=False, interpolation='bilinear', lazy=False),
            Flip(flip_ratio=.5),
            FormatShape(input_format='NCTHW'),
            PackActionInputs()]
# we note here multi_class is just to inform the class to compute onehot
dataset=VideoDataset(ann_file=args.ann_file, pipeline=pipeline, data_root=args.data_root, data_prefix=dict(video=args.data_prefix), multi_class=False, 
                 num_classes=None, start_index=0, modality='RGB', test_mode=False, delimiter=' ', lazy_init=False)
sampler=DefaultSampler(dataset, shuffle=True, seed=runner._seed, round_up=True)
init_fn=partial(worker_init_fn, num_workers=8, rank=0, seed=runner._seed)
runner._train_dataloader=DataLoader(dataset=dataset, batch_size=8, sampler=sampler, num_workers=8, persistent_workers=True,
                  collate_fn=pseudo_collate, worker_init_fn=init_fn)
runner._training_loop=EpochBasedTrainingLoop(runner, dataloader=runner._train_dataloader, max_epochs=256, val_begin=1, val_interval=5, dynamic_intervals=None)

[`build_optim_wrapper`](https://github.com/open-mmlab/mmengine/blob/main/mmengine/runner/runner.py#L987)

In [17]:
optimizer=torch.optim.SGD(runner.model.parameters(),lr=0.1, momentum=0.9, weight_decay=1e-4)
runner.optim_wrapper=OptimWrapper(optimizer=optimizer, clip_grad=dict(max_norm=40, norm_type=2))
runner.optim_wrapper._accumulative_counts 

1

build_param_scheduler

In [6]:
param_schedulers=[]
for scheduler in runner.cfg.param_scheduler:
    print(scheduler)
    _scheduler=copy.deepcopy(scheduler)
    default_end=runner._training_loop.max_epochs if _scheduler.get('by_epoch', True) else runner._training_loop.max_iters
    _scheduler.setdefault('end', default_end)
    _scheduler|=dict(optimizer=runner.optim_wrapper, epoch_length=len(runner._train_dataloader))
    schedule_type=_scheduler.pop('type')
    if schedule_type=='LinearLR': cls=LinearLR
    elif schedule_type=='CosineAnnealingLR': cls=CosineAnnealingLR
    convert_to_iter=_scheduler.pop('convert_to_iter_based', False)
    epoch_length=_scheduler.pop('epoch_length', None)
    scheduler=cls(**_scheduler)
    if convert_to_iter:
        assert isinstance(epoch_length, int)
        scheduler.build_iter_from_epoch(**_scheduler, epoch_length=epoch_length)
    param_schedulers.append(scheduler)
runner.param_schedulers=param_schedulers

{'type': 'LinearLR', 'start_factor': 0.1, 'by_epoch': True, 'begin': 0, 'end': 34, 'convert_to_iter_based': True}
{'type': 'CosineAnnealingLR', 'T_max': 256, 'eta_min': 0, 'by_epoch': True, 'begin': 0, 'end': 256}


In [7]:
# build_val_loop

pipeline=[DecordInit(io_backend='disk'),
            SampleFrames(clip_len=32, frame_interval=2, num_clips=1, test_mode=True),
            DecordDecode(),
            Resize(scale=(-1, 256), keep_ratio=True, interpolation='bilinear', lazy=False),
            CenterCrop(crop_size=224),
            FormatShape(input_format='NCTHW'),
            PackActionInputs()]
# we note here multi_class is just to inform the class to compute onehot
val_dataset=VideoDataset(ann_file=args.ann_file, pipeline=pipeline, data_root=args.data_root, data_prefix=dict(video=args.data_prefix), test_mode=True,
                     multi_class=False, num_classes=None, start_index=0, modality='RGB', delimiter=' ', lazy_init=False, 
                    indices=np.arange(0, 9000,3).tolist()) # 9586 is the original number of data
sampler=DefaultSampler(val_dataset, shuffle=False, seed=runner._seed, round_up=True)
init_fn=partial(worker_init_fn, num_workers=8, rank=0, seed=runner._seed)
runner._val_dataloader=DataLoader(dataset=val_dataset, batch_size=8, sampler=sampler, num_workers=8, persistent_workers=True,
                           collate_fn=pseudo_collate, worker_init_fn=init_fn)
runner._val_evaluator=Evaluator()
runner._val_loop=ValLoop(runner, dataloader=runner._val_dataloader, evaluator=runner._val_evaluator, fp16=False)

In [9]:
# initialize the model weights if the model has
runner.model.init_weights()

In [18]:

if not runner._has_loaded and runner._resume:
    # auto resume from the latest checkpoint
    resume_from=self.resume(runner.cfg.latest)

runner.optim_wrapper.initialize_count_status(runner.model, runner._training_loop.iter, runner._training_loop.max_iters)